In [1]:
from transformers.models.llama.tokenization_llama import LlamaTokenizer as HFTokenizer
from transformers import AutoTokenizer
import os
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"]="python"
import pandas as pd
from tqdm import tqdm
tqdm.pandas()

/home/huang717/.conda/envs/irm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer_path = 'meta-llama/Llama-2-7b-chat-hf'
local_path = ''
t1 = AutoTokenizer.from_pretrained(tokenizer_path)
t2 = HFTokenizer.from_pretrained(tokenizer_path)

In [3]:
type(t1),t1, t1.pad_token_id, t1.bos_token_id, t1.eos_token_id

(transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast,
 LlamaTokenizerFast(name_or_path='meta-llama/Llama-2-7b-chat-hf', vocab_size=32000, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>'}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
 	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 },
 None,
 1,
 2)

In [4]:
type(t2),t2, t2.pad_token_id, t2.bos_token_id, t2.eos_token_id

(transformers.models.llama.tokenization_llama.LlamaTokenizer,
 LlamaTokenizer(name_or_path='meta-llama/Llama-2-7b-chat-hf', vocab_size=32000, model_max_length=1000000000000000019884624838656, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>'}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
 	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 },
 None,
 1,
 2)

In [14]:
if type(t2) == AutoTokenizer:
    print("Equal!")

In [5]:
input_prompt = "The theory of relativity states that"
print('LlamaTokenizer')
print(t2(input_prompt, add_special_tokens=True).input_ids)
print('\nLlamaTokenizerFast')
print(t1(input_prompt, add_special_tokens=True).input_ids)

LlamaTokenizer
[1, 450, 6368, 310, 14215, 537, 5922, 393]

LlamaTokenizerFast
[1, 450, 6368, 310, 14215, 537, 5922, 393]


In [6]:
raw_data_path = "/home/huang717/DRAGN/IRM/injectable-alignment-model/datasets/anger_QA_7b_60k/split/anger_60k_val.csv"
df:pd.DataFrame = pd.read_csv(raw_data_path, dtype=str, na_filter=False)
df,df.shape

(     Unnamed: 0                                               text
 0         50862  What was the appearance of the Arcade model?  ...
 1         40032  Which two quadrants light up on the Xbox 360 S...
 2         48801  What calendar came into use in 153 BC?   Oh gr...
 3         30640  What is the Old Style date mean?   Grrrr... Of...
 4          8549  Whose reform included increasing the number of...
 ...         ...                                                ...
 2995       5657  How many ATI developers worked on the graphic ...
 2996       1814  In 2006, what was produced by Lifesize Comunic...
 2997      29554  When did Turkey start using the Gregorian cale...
 2998       1276  What did John Dee note?   Oh great, really? Yo...
 2999      28647  How many original Xboxes were sold in Japan be...
 
 [3000 rows x 2 columns],
 (3000, 2))

In [7]:
def tokenize_data_chunk(tokenizer, chunk):  
    """
    Tokenize a chunk of data using the given tokenizer.

    This function is highly dependent on the structure of the input data, and the tokenizer being used.
    """
    to_tokenize:str = chunk['text']

    # Does not pad during pre processing, pads dynamically during training
    result = tokenizer(to_tokenize, add_special_tokens=True, padding=False)
    chunk['Tokenized_Data'] = result.input_ids

    return chunk

In [8]:
tok_lambda1 = lambda x: tokenize_data_chunk(tokenizer=t1, chunk=x)  # 'df.' of line 62 becomes 'x' in this lambda
tok_lambda2 = lambda x: tokenize_data_chunk(tokenizer=t2, chunk=x)  # 'df.' of line 62 becomes 'x' in this lambda
print(f'Dataframe: {df}\n\n')

Dataframe:      Unnamed: 0                                               text
0         50862  What was the appearance of the Arcade model?  ...
1         40032  Which two quadrants light up on the Xbox 360 S...
2         48801  What calendar came into use in 153 BC?   Oh gr...
3         30640  What is the Old Style date mean?   Grrrr... Of...
4          8549  Whose reform included increasing the number of...
...         ...                                                ...
2995       5657  How many ATI developers worked on the graphic ...
2996       1814  In 2006, what was produced by Lifesize Comunic...
2997      29554  When did Turkey start using the Gregorian cale...
2998       1276  What did John Dee note?   Oh great, really? Yo...
2999      28647  How many original Xboxes were sold in Japan be...

[3000 rows x 2 columns]




In [9]:
df1 = df.progress_apply(tok_lambda1, axis=1)
df1,df1.shape

100%|██████████| 3000/3000 [00:02<00:00, 1243.86it/s]


(     Unnamed: 0                                               text  \
 0         50862  What was the appearance of the Arcade model?  ...   
 1         40032  Which two quadrants light up on the Xbox 360 S...   
 2         48801  What calendar came into use in 153 BC?   Oh gr...   
 3         30640  What is the Old Style date mean?   Grrrr... Of...   
 4          8549  Whose reform included increasing the number of...   
 ...         ...                                                ...   
 2995       5657  How many ATI developers worked on the graphic ...   
 2996       1814  In 2006, what was produced by Lifesize Comunic...   
 2997      29554  When did Turkey start using the Gregorian cale...   
 2998       1276  What did John Dee note?   Oh great, really? Yo...   
 2999      28647  How many original Xboxes were sold in Japan be...   
 
                                          Tokenized_Data  
 0     [1, 1724, 471, 278, 10097, 310, 278, 826, 6332...  
 1     [1, 8449, 1023, 15448

In [10]:
df2 = df.progress_apply(tok_lambda2, axis=1)
df2,df2.shape

100%|██████████| 3000/3000 [00:03<00:00, 863.76it/s]


(     Unnamed: 0                                               text  \
 0         50862  What was the appearance of the Arcade model?  ...   
 1         40032  Which two quadrants light up on the Xbox 360 S...   
 2         48801  What calendar came into use in 153 BC?   Oh gr...   
 3         30640  What is the Old Style date mean?   Grrrr... Of...   
 4          8549  Whose reform included increasing the number of...   
 ...         ...                                                ...   
 2995       5657  How many ATI developers worked on the graphic ...   
 2996       1814  In 2006, what was produced by Lifesize Comunic...   
 2997      29554  When did Turkey start using the Gregorian cale...   
 2998       1276  What did John Dee note?   Oh great, really? Yo...   
 2999      28647  How many original Xboxes were sold in Japan be...   
 
                                          Tokenized_Data  
 0     [1, 1724, 471, 278, 10097, 310, 278, 826, 6332...  
 1     [1, 8449, 1023, 15448

In [11]:
df1.drop(['text'], axis=1)

,Unnamed: 0,Tokenized_Data
0,50862,"[1, 1724, 471, 278, 10097, 310, 278, 826, 6332..."
1,40032,"[1, 8449, 1023, 15448, 1934, 3578, 701, 373, 2..."
2,48801,"[1, 1724, 17684, 2996, 964, 671, 297, 29871, 2..."
3,30640,"[1, 1724, 338, 278, 8198, 22135, 2635, 2099, 2..."
4,8549,"[1, 806, 852, 11736, 5134, 10231, 278, 1353, 3..."
...,...,...
2995,5657,"[1, 1128, 1784, 15531, 29902, 18777, 3796, 373..."
2996,1814,"[1, 512, 29871, 29906, 29900, 29900, 29953, 29..."
2997,29554,"[1, 1932, 1258, 26459, 1369, 773, 278, 12051, ..."
2998,1276,"[1, 1724, 1258, 2259, 897, 29872, 4443, 29973,..."


In [12]:
df2.drop(['text'], axis=1)

,Unnamed: 0,Tokenized_Data
0,50862,"[1, 1724, 471, 278, 10097, 310, 278, 826, 6332..."
1,40032,"[1, 8449, 1023, 15448, 1934, 3578, 701, 373, 2..."
2,48801,"[1, 1724, 17684, 2996, 964, 671, 297, 29871, 2..."
3,30640,"[1, 1724, 338, 278, 8198, 22135, 2635, 2099, 2..."
4,8549,"[1, 806, 852, 11736, 5134, 10231, 278, 1353, 3..."
...,...,...
2995,5657,"[1, 1128, 1784, 15531, 29902, 18777, 3796, 373..."
2996,1814,"[1, 512, 29871, 29906, 29900, 29900, 29953, 29..."
2997,29554,"[1, 1932, 1258, 26459, 1369, 773, 278, 12051, ..."
2998,1276,"[1, 1724, 1258, 2259, 897, 29872, 4443, 29973,..."


In [13]:
path_to_data = "/home/huang717/DRAGN/IRM/injectable-alignment-model/datasets/anger_QA_7b_60k/tokenized/anger_60k_val.pkl"
data = pd.read_pickle(path_to_data)
print("DataFrame info:")
print(data.info())
print("\nFirst few rows:")
print(data.head())
print(data.shape)
print(data.columns)

DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 1 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  3000 non-null   object
dtypes: object(1)
memory usage: 23.6+ KB
None

First few rows:
  Unnamed: 0
0      50862
1      40032
2      48801
3      30640
4       8549
(3000, 1)
Index(['Unnamed: 0'], dtype='object')
